# M12 · Two-tower / EBR retrieval architecture

_Curriculum · Domain 2 · Retrieval & Representation_

**Train query and item towers into one space, then serve with precomputed item embeddings.**

The batch loss is $\ell=-\log\frac{\exp(s_{+})}{\sum_j\exp(s_j)}$, and the serving score is a dot product.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)

## Tiny batch of query and item embeddings

Rows are matched pairs. Every other item in the batch becomes an in-batch negative.

In [ ]:
query = np.array([
    [1.0, 0.2, 0.1],
    [0.1, 1.0, 0.2],
    [0.2, 0.1, 1.0],
    [0.8, 0.4, 0.1],
])
item = np.array([
    [0.9, 0.3, 0.1],
    [0.0, 0.9, 0.3],
    [0.3, 0.2, 0.9],
    [0.7, 0.5, 0.0],
])

scores = query @ item.T

print(pd.DataFrame(scores).round(3))

## Step 1 - Softmax over each row

The diagonal is positive. The off-diagonal entries are negatives supplied by the batch.

In [ ]:
shifted = scores - scores.max(axis=1, keepdims=True)
exp_scores = np.exp(shifted)
probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
pos_probs = np.diag(probs)
loss = -np.log(pos_probs).mean()

print("positive probabilities:", pos_probs.round(3))
print("mean loss:", round(loss, 3))

assert pos_probs.shape[0] == 4

## Step 2 - Apply a logQ correction

If frequent items are sampled more often, subtracting $\log Q(i)$ changes the logits used by the sampled-softmax loss.

In [ ]:
sample_q = np.array([0.50, 0.20, 0.20, 0.10])
corrected_scores = scores - np.log(sample_q)[None, :]
corrected_shifted = corrected_scores - corrected_scores.max(axis=1, keepdims=True)
corrected_exp = np.exp(corrected_shifted)
corrected_probs = corrected_exp / corrected_exp.sum(axis=1, keepdims=True)
corrected_loss = -np.log(np.diag(corrected_probs)).mean()

print("corrected positive probabilities:", np.diag(corrected_probs).round(3))
print("corrected loss:", round(corrected_loss, 3))

assert corrected_loss > 0

## Step 3 - Serve by searching item embeddings

At serving time, the item tower has already run. A fresh query vector is compared to cached item vectors.

In [ ]:
catalog = np.vstack([item, rng.normal(scale=0.4, size=(8, 3))])
creator_names = np.array([f"creator_{i}" for i in range(catalog.shape[0])])
serve_query = np.array([0.85, 0.35, 0.05])
serve_scores = catalog @ serve_query
top_idx = np.argsort(-serve_scores)[:5]

print(pd.DataFrame({"creator": creator_names[top_idx], "score": serve_scores[top_idx]}).round(3))

assert top_idx[0] in [0, 3]

## Visualize the training score matrix

Good retrieval training pushes the diagonal above the off-diagonal entries.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(scores, cmap="Blues")
ax.set_xlabel("item in batch")
ax.set_ylabel("query in batch")
ax.set_title("two-tower scores")
fig.colorbar(im, ax=ax)
plt.show()

## Practice

Change `sample_q` so one negative is much rarer. Observe how the corrected probability and loss move.

In [ ]:
# Your turn:
